# OpsTwin Recovery Arena: Minimal SFT Training Notebook

**Meta PyTorch OpenEnv Hackathon, Theme 3.1 Professional Tasks**  
**Model:** Qwen3-0.6B · **Trainer:** HF TRL `SFTTrainer` · **Runtime target:** Colab Free (T4 GPU, 16 GB VRAM)

This notebook is the **minimum reproducible training script** required by the hackathon rubric. It does three things:

1. Loads the OpsTwin environment and a 200-example slice of our expert-trajectory SFT dataset.
2. Fine-tunes `Qwen/Qwen3-0.6B` with LoRA for 30 training steps using `trl.SFTTrainer`.
3. Plots the training-loss curve and runs one before/after rollout against the `bad_release` scenario.

The full training run (Qwen3-1.7B, 2 epochs, ~3,600 trajectories) lives in `train_sft_5090.py` in the repo. That produces [`Deltasthic/opstwin-qwen3-1.7b-sft`](https://huggingface.co/Deltasthic/opstwin-qwen3-1.7b-sft). This notebook is the Colab-friendly shrunken version for judges and reviewers who want to execute the pipeline themselves.

## 1. Setup: install dependencies

Runs in ~3 min on Colab Free. If you hit a CUDA version mismatch, restart the runtime and re-run this cell.

In [ ]:
!pip install -q \
    "openenv-core>=0.2.2" \
    "trl>=0.14" \
    "transformers>=4.45" \
    "peft>=0.13" \
    "datasets>=2.18" \
    "accelerate>=0.30" \
    "fastapi>=0.104.0" \
    "uvicorn>=0.24.0" \
    "pydantic>=2.0.0" \
    "matplotlib>=3.8" \
    "openai>=1.0.0" \
    "python-dotenv>=1.0"

## 2. Clone the environment code

The environment definition (`models.py`, `server/`, `baselines/`, `sft_dataset.jsonl`) lives in the public GitHub repo. We clone it so the notebook can import `OpsTwinEnvironment` directly.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/Deltasthicc/opstwin-recovery.git"
REPO_DIR = "/content/opstwin-recovery"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("Repo at:", os.getcwd())
print("Files:", [f for f in os.listdir('.') if not f.startswith('.')][:15])

## 3. Sanity-check the environment

Before training, confirm the environment boots and a single episode plays end-to-end. This is the same check `tests/smoke_http.py` does over HTTP, but in-process here for speed.

In [ ]:
from models import OpsAction
from server.environment import OpsTwinEnvironment

env = OpsTwinEnvironment()
obs = env.reset(task="bad_release")
print(f"Scenario: {obs.ops_status}")
print(f"Services: {len(obs.services)}  Tickets: {len(obs.tickets)}  Max steps: {env._max_steps}")

for cmd in ["SWITCH_DESK INCIDENT_COMMAND", "ASSESS_BLAST_RADIUS", "DONE"]:
    obs = env.step(OpsAction(command=cmd))
    print(f"  {cmd:35s} reward={obs.reward:+.3f}  done={obs.done}")
print(f"Final score: {obs.score}")

## 4. Load the SFT dataset

`sft_dataset.jsonl` holds ~3,600 expert-trajectory examples (ChatML conversations with `system`/`user`/`assistant` roles). For Colab Free we use 200 examples to keep training under 5 minutes.

In [ ]:
import json, random
from datasets import Dataset

random.seed(42)

examples = []
with open("sft_dataset.jsonl") as f:
    for line in f:
        examples.append(json.loads(line))

print(f"Total expert examples available: {len(examples)}")

SUBSET = 200
random.shuffle(examples)
examples = examples[:SUBSET]

split = int(0.9 * len(examples))
train_ds = Dataset.from_list(examples[:split])
eval_ds = Dataset.from_list(examples[split:])
print(f"Train: {len(train_ds)}  Eval: {len(eval_ds)}")

print("\nExample assistant target:")
print("  ", examples[0]["messages"][-1]["content"][:100])

## 5. Load Qwen3-0.6B with LoRA

Small enough to fit on a T4 with `bf16`. LoRA rank 8 keeps trainable params under 10 M.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Train for 30 steps with `SFTTrainer`

This cell is the hackathon rubric's required artifact: **a TRL training script runnable in Colab.** It should finish in under 5 minutes on a T4 and show training loss trending down.

In [ ]:
from trl import SFTConfig, SFTTrainer

config = SFTConfig(
    output_dir="/tmp/opstwin-colab-checkpoints",
    max_steps=30,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=False,
    optim="adamw_torch",
    max_length=2048,
    packing=False,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

trainer.train()

## 7. Plot the training-loss curve

The rubric explicitly asks for "observable evidence of training progress". This curve is what judges look for.

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_steps = [h["step"] for h in history if "loss" in h]
train_loss = [h["loss"] for h in history if "loss" in h]
eval_steps = [h["step"] for h in history if "eval_loss" in h]
eval_loss = [h["eval_loss"] for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 4))
plt.plot(train_steps, train_loss, marker="o", label="train loss")
if eval_loss:
    plt.plot(eval_steps, eval_loss, marker="s", label="eval loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("OpsTwin SFT training (Qwen3-0.6B + LoRA)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/opstwin_training_curve.png", dpi=120)
plt.show()
print(f"Start train loss: {train_loss[0]:.3f}")
print(f"End   train loss: {train_loss[-1]:.3f}")

## 8. Before/after rollout on the `bad_release` scenario

A one-episode qualitative demo: play `bad_release` with the trained policy, print the action trace and final score. The baseline (untrained Qwen3-0.6B) scores ~0.18 on this scenario; the full Qwen3-1.7B fine-tune scores ~0.42. Expect a shrunken version of that improvement here.

In [ ]:
from inference import SYSTEM_PROMPT, build_user_prompt

def rollout(policy_model, scenario="bad_release", max_steps=14):
    env = OpsTwinEnvironment()
    obs = env.reset(task=scenario)
    history, prev_reward = [], 0.0
    trace = []
    for step in range(1, max_steps + 1):
        user_prompt = build_user_prompt(step, obs, prev_reward, history)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
        inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(policy_model.device)
        with torch.no_grad():
            out = policy_model.generate(inputs, max_new_tokens=32, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        cmd = tokenizer.decode(out[0, inputs.shape[-1]:], skip_special_tokens=True).strip().splitlines()[0]
        obs = env.step(OpsAction(command=cmd))
        trace.append((step, cmd, obs.reward))
        history.append(f"S{step}: {cmd} -> {obs.reward:+.2f}")
        history = history[-5:]
        prev_reward = obs.reward
        if obs.done:
            break
    return trace, obs.score

print("=== TRAINED ROLLOUT (bad_release) ===")
trace, final_score = rollout(trainer.model, "bad_release")
for step, cmd, reward in trace:
    print(f"  S{step:2d}  {cmd:40s} reward={reward:+.3f}")
print(f"Final score: {final_score:.3f}")

## 9. What's next

This minimal run demonstrates the pipeline end-to-end. To reproduce the full hackathon result:

- Switch `MODEL_ID` to `Qwen/Qwen3-1.7B`.
- Remove the `SUBSET = 200` cap to use all ~3,600 trajectories.
- Bump `max_steps` (or use `num_train_epochs=2`), disable LoRA for full fine-tune, and use a higher-memory GPU (RTX 5090 or A100).
- See `train_sft_5090.py` in the repo. That is the script behind our published model [`Deltasthic/opstwin-qwen3-1.7b-sft`](https://huggingface.co/Deltasthic/opstwin-qwen3-1.7b-sft).

Held-out evaluation across 5 seeds lives in `evaluate.py`.